
# Selective Refinement — Global Compatibility Gate

This CPU-only notebook tests the next question after the local-node method gate:

> Can all local hierarchical sensitivity constraints be represented by **one globally compatible leaf distribution and one valid collection of monotone refinement trajectories**?

The gate uses a depth-3 synthetic tree and checks:

1. a valid monotone tree-wide refinement mechanism;
2. an explicit **trajectory-flow LP** with latent leaf/trajectory assignment variables;
3. a much smaller **subtree-mass LP** using unconditional leaf masses;
4. equality of the two formulations on every leaf endpoint and multiple random linear objectives;
5. tightening from multiple observation horizons;
6. robust-risk improvement over an independently assembled marginal-interval relaxation;
7. computational scaling to 256 leaves.

A pass is **not** a publication-level novelty claim. It would justify moving to a theorem/proof gate for arbitrary trees.


In [1]:

import json, math, time, zipfile, os
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog

SEED = 20261004
GAMMA = 4.0
H = 3
OUTDIR = Path("selective_refinement_global_compatibility_gate_results")
OUTDIR.mkdir(exist_ok=True)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Gamma:", GAMMA, "Horizons:", H)


NumPy: 2.3.5
Pandas: 2.2.3
Gamma: 4.0 Horizons: 3


## 1. Build a depth-3 hierarchy and a valid informative refinement mechanism

In [2]:

children = {
    "ROOT": ["A","B","C"],
    "A": ["A1","A2","A3"],
    "B": ["B1","B2"],
    "C": ["C1","C2","C3"],
    "A1":["A1a","A1b"], "A2":["A2a","A2b"], "A3":["A3a","A3b"],
    "B1":["B1a","B1b"], "B2":["B2a","B2b"],
    "C1":["C1a","C1b"], "C2":["C2a","C2b"], "C3":["C3a","C3b"],
}

internal = set(children)
all_nodes = set(children)
for ch in children.values():
    all_nodes.update(ch)
leaves = sorted(all_nodes - internal)

parent = {}
for u, chs in children.items():
    for c in chs:
        parent[c] = u

def path_to_root(node):
    p = [node]
    while p[-1] != "ROOT":
        p.append(parent[p[-1]])
    return list(reversed(p))

def descendant_leaves(node):
    if node not in children:
        return [node]
    out = []
    for c in children[node]:
        out.extend(descendant_leaves(c))
    return out

subleaves = {n: descendant_leaves(n) for n in all_nodes}

# Cumulative probability that each edge has been crossed by horizons 1,2,3.
# Each descendant edge is no faster than its parent edge.
q_edge = {
    "A":[.18,.42,.75], "B":[.40,.60,.75], "C":[.28,.50,.75],
    "A1":[.05,.25,.60], "A2":[.13,.35,.62], "A3":[.08,.30,.61],
    "B1":[.15,.38,.62], "B2":[.28,.50,.64],
    "C1":[.09,.30,.60], "C2":[.19,.42,.63], "C3":[.12,.34,.61],
}

# Leaf edges inherit a slower cumulative refinement curve.
for node in list(children):
    if node in ["ROOT","A","B","C"]:
        continue
    base = np.asarray(q_edge[node], dtype=float)
    for idx, c in enumerate(children[node]):
        factor = [0.58, 0.82][idx]
        q_edge[c] = (base * factor).tolist()

# Check parent-before-descendant ordering.
for node, chs in children.items():
    if node == "ROOT":
        continue
    for c in chs:
        assert np.all(np.asarray(q_edge[c]) <= np.asarray(q_edge[node]) + 1e-12)

def odds(q):
    return q / (1.0 - q)

max_true_or = 0.0
or_rows = []
for u, chs in children.items():
    for h in range(H):
        oo = np.array([odds(q_edge[c][h]) for c in chs])
        ratio = float(oo.max() / oo.min())
        max_true_or = max(max_true_or, ratio)
        or_rows.append({"node":u, "horizon":h+1, "max_sibling_odds_ratio":ratio})

assert max_true_or <= GAMMA + 1e-12

rng = np.random.default_rng(SEED)
raw = rng.gamma(2.0, 1.0, len(leaves))
p_true = raw / raw.sum()
ptrue = dict(zip(leaves, p_true))

print("Leaves:", len(leaves))
print("Internal nodes:", len(children))
print("Maximum true sibling odds ratio:", max_true_or)


Leaves: 16
Internal nodes: 12
Maximum true sibling odds ratio: 3.0370370370370376



We generate exact trajectory probabilities using a shared latent \(U\sim \mathrm{Uniform}(0,1)\).
For a given leaf, edge \(e\) is crossed by horizon \(h\) when \(U\le q_e(h)\).
Because descendant cumulative probabilities never exceed ancestor probabilities, every generated trajectory is automatically monotone and parent-before-child.


In [3]:

group_mass = defaultdict(float)

for leaf in leaves:
    path = path_to_root(leaf)
    thresholds = sorted(set(
        [0.0, 1.0] +
        [q_edge[n][h] for n in path[1:] for h in range(H)]
    ))
    for a, b in zip(thresholds[:-1], thresholds[1:]):
        if b - a <= 1e-14:
            continue
        U = 0.5 * (a + b)
        states = []
        for h in range(H):
            state = "ROOT"
            for n in path[1:]:
                if U <= q_edge[n][h] + 1e-15:
                    state = n
                else:
                    break
            states.append(state)
        group_mass[tuple(states)] += ptrue[leaf] * (b - a)

assert abs(sum(group_mass.values()) - 1.0) < 1e-12

# Verify each observed trajectory is monotone down one tree path.
def is_ancestor(a, b):
    return a in path_to_root(b)

for g in group_mass:
    for s1, s2 in zip(g[:-1], g[1:]):
        assert is_ancestor(s1, s2)

print("Distinct observed trajectory groups:", len(group_mass))
print("Total mass:", sum(group_mass.values()))


Distinct observed trajectory groups: 121
Total mass: 1.0000000000000002


## 2. Recover observable edge-crossing masses

In [4]:

# r[u][c][h] = observable mass that has crossed edge u->c by horizon h.
r = defaultdict(lambda: defaultdict(lambda: [0.0] * H))

for g, mass in group_mass.items():
    for h, state in enumerate(g):
        state_path = set(path_to_root(state))
        for u, chs in children.items():
            for c in chs:
                if c in state_path:
                    r[u][c][h] += mass

# Confirm exact agreement with the data-generating mechanism.
max_r_error = 0.0
for u, chs in children.items():
    for c in chs:
        true_subtree_mass = sum(ptrue[l] for l in subleaves[c])
        expected = true_subtree_mass * np.asarray(q_edge[c])
        max_r_error = max(max_r_error, float(np.max(np.abs(np.asarray(r[u][c]) - expected))))

assert max_r_error < 1e-12
print("Maximum crossing-mass reconstruction error:", max_r_error)


Maximum crossing-mass reconstruction error: 5.551115123125783e-17



## 3. Compact global subtree-mass LP

Let \(p_\ell=P(Y=\ell)\) be the unknown leaf probabilities and
\(s_c=\sum_{\ell\in\mathrm{subtree}(c)}p_\ell\).

For siblings \(j,k\) at node \(u\), the horizon-\(h\) refinement odds are

\[
\frac{r_{ujh}}{s_j-r_{ujh}},\qquad
\frac{r_{ukh}}{s_k-r_{ukh}}.
\]

The sensitivity restriction

\[
\frac{\mathrm{odds}_{j}}{\mathrm{odds}_{k}}\le \Gamma
\]

rearranges to the **linear** inequality

\[
r_j s_k-\Gamma r_k s_j \le (1-\Gamma)r_jr_k.
\]

Thus a single set of unconditional leaf masses can enforce every node and every horizon simultaneously.


In [5]:

leaf_index = {l:i for i,l in enumerate(leaves)}
nL = len(leaves)
Aeq = np.ones((1, nL))
beq = np.array([1.0])
bounds = [(0, None)] * nL

def subtree_vector(node):
    v = np.zeros(nL)
    for l in subleaves[node]:
        v[leaf_index[l]] = 1.0
    return v

def build_compact_constraints(horizons=None, include_gamma=True):
    if horizons is None:
        horizons = list(range(H))
    A, b = [], []

    # Candidate-set / trajectory compatibility:
    # anything already observed inside child subtree c must truly belong to c.
    for u, chs in children.items():
        for c in chs:
            A.append(-subtree_vector(c))
            b.append(-r[u][c][-1])

    if include_gamma:
        for u, chs in children.items():
            vec = {c: subtree_vector(c) for c in chs}
            for h in horizons:
                for j in chs:
                    for k in chs:
                        if j == k:
                            continue
                        rj, rk = r[u][j][h], r[u][k][h]
                        A.append(rj * vec[k] - GAMMA * rk * vec[j])
                        b.append((1.0 - GAMMA) * rj * rk)

    return np.asarray(A), np.asarray(b)

A_all, b_all = build_compact_constraints()
feas = linprog(
    np.zeros(nL), A_ub=A_all, b_ub=b_all,
    A_eq=Aeq, b_eq=beq, bounds=bounds, method="highs"
)
assert feas.success
truth_max_violation = float(np.max(A_all @ p_true - b_all))
assert truth_max_violation <= 1e-10

print("Compact variables:", nL)
print("Compact inequalities:", len(b_all))
print("Truth max constraint violation:", truth_max_violation)


Compact variables: 16
Compact inequalities: 135
Truth max constraint violation: -3.1004804058557023e-05


## 4. Explicit trajectory-flow LP

In [6]:

groups = list(group_mass)

def compatible_trajectory_leaf(g, leaf):
    anc = set(path_to_root(leaf))
    return all(state in anc for state in g)

pairs = [
    (gi, leaf)
    for gi, g in enumerate(groups)
    for leaf in leaves
    if compatible_trajectory_leaf(g, leaf)
]

nZ = len(pairs)
Pmat = np.zeros((nL, nZ))
Aeq_flow = np.zeros((len(groups), nZ))
beq_flow = np.array([group_mass[g] for g in groups])

for zi, (gi, leaf) in enumerate(pairs):
    Pmat[leaf_index[leaf], zi] = 1.0
    Aeq_flow[gi, zi] = 1.0

A_flow = A_all @ Pmat
b_flow = b_all.copy()
flow_bounds = [(0, None)] * nZ

flow_feas = linprog(
    np.zeros(nZ), A_ub=A_flow, b_ub=b_flow,
    A_eq=Aeq_flow, b_eq=beq_flow,
    bounds=flow_bounds, method="highs"
)
assert flow_feas.success

# Validate the reconstructed joint mechanism.
z = flow_feas.x
row_resid = np.max(np.abs(Aeq_flow @ z - beq_flow))
p_from_flow = Pmat @ z
assert row_resid < 1e-9
assert abs(p_from_flow.sum() - 1.0) < 1e-9

print("Trajectory groups:", len(groups))
print("Trajectory-flow variables:", nZ)
print("Max trajectory-group mass residual:", row_resid)


Trajectory groups: 121
Trajectory-flow variables: 170
Max trajectory-group mass residual: 6.938893903907228e-18


## 5. Sharp computational equivalence: full flow LP vs compact LP

In [7]:

def compact_endpoint(leaf):
    c = np.zeros(nL)
    c[leaf_index[leaf]] = 1.0
    lo = linprog(c, A_ub=A_all, b_ub=b_all, A_eq=Aeq, b_eq=beq, bounds=bounds, method="highs")
    hi = linprog(-c, A_ub=A_all, b_ub=b_all, A_eq=Aeq, b_eq=beq, bounds=bounds, method="highs")
    assert lo.success and hi.success
    return lo.fun, -hi.fun

def flow_endpoint(leaf):
    c = Pmat[leaf_index[leaf]]
    lo = linprog(c, A_ub=A_flow, b_ub=b_flow, A_eq=Aeq_flow, b_eq=beq_flow, bounds=flow_bounds, method="highs")
    hi = linprog(-c, A_ub=A_flow, b_ub=b_flow, A_eq=Aeq_flow, b_eq=beq_flow, bounds=flow_bounds, method="highs")
    assert lo.success and hi.success
    return lo.fun, -hi.fun

eq_rows = []
max_endpoint_difference = 0.0
for leaf in leaves:
    clo, chi = compact_endpoint(leaf)
    flo, fhi = flow_endpoint(leaf)
    diff = max(abs(clo-flo), abs(chi-fhi))
    max_endpoint_difference = max(max_endpoint_difference, diff)
    eq_rows.append({
        "leaf":leaf, "truth":ptrue[leaf],
        "compact_lower":clo, "compact_upper":chi,
        "flow_lower":flo, "flow_upper":fhi,
        "max_endpoint_difference":diff
    })

equivalence_df = pd.DataFrame(eq_rows)

# Stronger check: compare support functions in 20 random directions.
rng_obj = np.random.default_rng(123)
objective_rows = []
max_objective_difference = 0.0
for k in range(20):
    v = rng_obj.normal(size=nL)
    rc = linprog(-v, A_ub=A_all, b_ub=b_all, A_eq=Aeq, b_eq=beq, bounds=bounds, method="highs")
    rf = linprog(-(v @ Pmat), A_ub=A_flow, b_ub=b_flow, A_eq=Aeq_flow, b_eq=beq_flow, bounds=flow_bounds, method="highs")
    assert rc.success and rf.success
    compact_value, flow_value = -rc.fun, -rf.fun
    d = abs(compact_value-flow_value)
    max_objective_difference = max(max_objective_difference, d)
    objective_rows.append({
        "objective_id":k,
        "compact_max":compact_value,
        "flow_max":flow_value,
        "absolute_difference":d
    })

objective_df = pd.DataFrame(objective_rows)

assert max_endpoint_difference < 1e-9
assert max_objective_difference < 1e-9

print("Max leaf-endpoint difference:", max_endpoint_difference)
print("Max random-objective difference:", max_objective_difference)


Max leaf-endpoint difference: 4.996003610813204e-16
Max random-objective difference: 5.551115123125783e-16


## 6. Do hierarchy and multiple horizons tighten identification?

In [8]:

def endpoint_table(A, b, label):
    rows = []
    for leaf in leaves:
        c = np.zeros(nL)
        c[leaf_index[leaf]] = 1.0
        lo = linprog(c, A_ub=A, b_ub=b, A_eq=Aeq, b_eq=beq, bounds=bounds, method="highs")
        hi = linprog(-c, A_ub=A, b_ub=b, A_eq=Aeq, b_eq=beq, bounds=bounds, method="highs")
        assert lo.success and hi.success
        rows.append({
            "leaf":leaf, "truth":ptrue[leaf],
            f"{label}_lower":lo.fun,
            f"{label}_upper":-hi.fun,
            f"{label}_width":-hi.fun-lo.fun
        })
    return pd.DataFrame(rows)

A_hier, b_hier = build_compact_constraints(include_gamma=False)
A_last, b_last = build_compact_constraints(horizons=[H-1], include_gamma=True)

hier_df = endpoint_table(A_hier, b_hier, "hierarchy")
last_df = endpoint_table(A_last, b_last, "last")
all_df = endpoint_table(A_all, b_all, "all")

bounds_df = hier_df.merge(last_df, on=["leaf","truth"]).merge(all_df, on=["leaf","truth"])

mean_hierarchy_width = float(bounds_df.hierarchy_width.mean())
mean_last_width = float(bounds_df.last_width.mean())
mean_all_width = float(bounds_df.all_width.mean())

multi_horizon_reduction = 1.0 - mean_all_width / mean_last_width
gamma_vs_hierarchy_reduction = 1.0 - mean_all_width / mean_hierarchy_width

all_not_wider = bool(np.all(bounds_df.all_width <= bounds_df.last_width + 1e-10))
at_least_one_strict = bool(np.any(bounds_df.all_width < bounds_df.last_width - 1e-6))
truth_contained = bool(np.all(
    (bounds_df.truth >= bounds_df.all_lower - 1e-10) &
    (bounds_df.truth <= bounds_df.all_upper + 1e-10)
))

assert all_not_wider
assert at_least_one_strict
assert truth_contained

print("Mean hierarchy-only width:", mean_hierarchy_width)
print("Mean last-horizon width:", mean_last_width)
print("Mean all-horizon width:", mean_all_width)
print("Relative all-vs-last reduction:", multi_horizon_reduction)
print("Relative all-vs-hierarchy reduction:", gamma_vs_hierarchy_reduction)


Mean hierarchy-only width: 0.31963059267737526
Mean last-horizon width: 0.11484074798142302
Mean all-horizon width: 0.10966039454569984
Relative all-vs-last reduction: 0.04510901859121619
Relative all-vs-hierarchy reduction: 0.6569152106901498


## 7. Robust loss: global compatibility vs independent interval assembly

In [9]:

# A fixed, transparent classifier: probability increases linearly with leaf index.
weights = np.arange(1, nL+1, dtype=float)
pred = weights / weights.sum()
loss = -np.log(pred)

true_loss = float(loss @ p_true)

last_res = linprog(-loss, A_ub=A_last, b_ub=b_last, A_eq=Aeq, b_eq=beq, bounds=bounds, method="highs")
all_res = linprog(-loss, A_ub=A_all, b_ub=b_all, A_eq=Aeq, b_eq=beq, bounds=bounds, method="highs")
assert last_res.success and all_res.success

last_robust_upper = float(-last_res.fun)
global_robust_upper = float(-all_res.fun)

# "Independent assembly": retain each globally valid leaf projection interval,
# but discard dependencies between leaves except sum-to-one.
box_bounds = list(zip(bounds_df.all_lower.to_numpy(), bounds_df.all_upper.to_numpy()))
box_res = linprog(-loss, A_eq=Aeq, b_eq=beq, bounds=box_bounds, method="highs")
assert box_res.success
independent_interval_upper = float(-box_res.fun)

assert global_robust_upper + 1e-10 >= true_loss
assert global_robust_upper <= last_robust_upper + 1e-10
assert global_robust_upper < independent_interval_upper - 1e-6

loss_df = pd.DataFrame([{
    "true_loss": true_loss,
    "last_horizon_robust_upper": last_robust_upper,
    "all_horizon_global_robust_upper": global_robust_upper,
    "independent_interval_upper": independent_interval_upper,
    "all_vs_last_absolute_tightening": last_robust_upper-global_robust_upper,
    "global_vs_independent_absolute_tightening": independent_interval_upper-global_robust_upper,
    "global_vs_independent_relative_tightening": 1.0-global_robust_upper/independent_interval_upper
}])

print(loss_df.to_string(index=False))


 true_loss  last_horizon_robust_upper  all_horizon_global_robust_upper  independent_interval_upper  all_vs_last_absolute_tightening  global_vs_independent_absolute_tightening  global_vs_independent_relative_tightening
  2.925939                   3.105689                           3.0815                    3.292552                         0.024188                                   0.211052                                     0.0641


## 8. Compact-LP scaling audit

In [10]:

def scaling_problem(branches_n, leaves_per_branch, H=3, gamma=GAMMA):
    rng = np.random.default_rng(SEED + branches_n*100 + leaves_per_branch)
    n = branches_n * leaves_per_branch
    p = rng.dirichlet(np.ones(n)*2.0)
    branch_mass = p.reshape(branches_n, leaves_per_branch).sum(axis=1)

    base = np.array([0.28,0.52,0.76])
    qroot = np.zeros((branches_n,H))
    for i in range(branches_n):
        mult = 0.85 + 0.30 * (i / max(1, branches_n-1))
        qroot[i] = np.minimum(base*mult, 0.90)

    qleaf = np.zeros((branches_n, leaves_per_branch, H))
    for i in range(branches_n):
        for j in range(leaves_per_branch):
            factor = 0.52 + 0.26 * (j / max(1, leaves_per_branch-1))
            qleaf[i,j] = qroot[i] * factor

    A, b = [], []

    # final compatibility lower bounds
    for i in range(branches_n):
        v = np.zeros(n)
        v[i*leaves_per_branch:(i+1)*leaves_per_branch] = -1
        A.append(v)
        b.append(-branch_mass[i]*qroot[i,-1])

        for j in range(leaves_per_branch):
            vv = np.zeros(n)
            vv[i*leaves_per_branch+j] = -1
            A.append(vv)
            b.append(-p[i*leaves_per_branch+j]*qleaf[i,j,-1])

    # root sibling gamma constraints
    root_vec = []
    for i in range(branches_n):
        v = np.zeros(n)
        v[i*leaves_per_branch:(i+1)*leaves_per_branch] = 1
        root_vec.append(v)

    for h in range(H):
        rr = branch_mass*qroot[:,h]
        for i in range(branches_n):
            for j in range(branches_n):
                if i == j:
                    continue
                A.append(rr[i]*root_vec[j] - gamma*rr[j]*root_vec[i])
                b.append((1-gamma)*rr[i]*rr[j])

    # within-branch sibling gamma constraints
    for i in range(branches_n):
        for h in range(H):
            rr = p[i*leaves_per_branch:(i+1)*leaves_per_branch] * qleaf[i,:,h]
            for j in range(leaves_per_branch):
                for k in range(leaves_per_branch):
                    if j == k:
                        continue
                    v = np.zeros(n)
                    v[i*leaves_per_branch+k] += rr[j]
                    v[i*leaves_per_branch+j] += -gamma*rr[k]
                    A.append(v)
                    b.append((1-gamma)*rr[j]*rr[k])

    A, b = np.asarray(A), np.asarray(b)
    eq = np.ones((1,n))
    obj = np.linspace(0.2,2.0,n)

    t0 = time.perf_counter()
    res = linprog(-obj, A_ub=A, b_ub=b, A_eq=eq, b_eq=[1.0],
                  bounds=[(0,None)]*n, method="highs")
    elapsed = time.perf_counter()-t0

    return {
        "branches":branches_n,
        "leaves_per_branch":leaves_per_branch,
        "leaves":n,
        "inequalities":len(b),
        "solve_seconds":elapsed,
        "success":bool(res.success)
    }

scale_rows = []
for b, k in [(4,4),(4,8),(8,8),(8,16),(16,16)]:
    scale_rows.append(scaling_problem(b,k))

scaling_df = pd.DataFrame(scale_rows)
assert scaling_df.success.all()
assert scaling_df.iloc[-1].solve_seconds < 5.0

print(scaling_df.to_string(index=False))


 branches  leaves_per_branch  leaves  inequalities  solve_seconds  success
        4                  4      16           200       0.002220     True
        4                  8      32           744       0.002836     True
        8                  8      64          1584       0.006634     True
        8                 16     128          6064       0.031039     True
       16                 16     256         12512       0.111906     True


## 9. Save audit artifacts and issue the gate verdict

In [11]:

equivalence_df.to_csv(OUTDIR/"full_vs_compact_leaf_bounds.csv", index=False)
objective_df.to_csv(OUTDIR/"support_function_equivalence.csv", index=False)
bounds_df.to_csv(OUTDIR/"global_leaf_bounds.csv", index=False)
loss_df.to_csv(OUTDIR/"robust_loss_audit.csv", index=False)
scaling_df.to_csv(OUTDIR/"scaling_audit.csv", index=False)
pd.DataFrame(or_rows).to_csv(OUTDIR/"true_mechanism_gamma_audit.csv", index=False)

# Plot interval widths.
plt.figure(figsize=(9,4.5))
x = np.arange(len(leaves))
plt.plot(x, bounds_df.hierarchy_width, marker="o", label="Hierarchy only")
plt.plot(x, bounds_df.last_width, marker="o", label="Last horizon")
plt.plot(x, bounds_df.all_width, marker="o", label="All horizons")
plt.xticks(x, leaves, rotation=70)
plt.ylabel("Identified interval width")
plt.title("Global leaf-probability identification")
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR/"global_interval_widths.png", dpi=160)
plt.close()

# Plot scaling.
plt.figure(figsize=(6.5,4.2))
plt.plot(scaling_df.leaves, scaling_df.solve_seconds, marker="o")
plt.xlabel("Number of leaves")
plt.ylabel("LP solve time (seconds)")
plt.title("Compact global LP scaling")
plt.tight_layout()
plt.savefig(OUTDIR/"global_lp_scaling.png", dpi=160)
plt.close()

checks = {
    "monotone_tree_trajectories": True,
    "true_mechanism_within_gamma": bool(max_true_or <= GAMMA + 1e-12),
    "truth_feasible_in_global_lp": bool(truth_max_violation <= 1e-10),
    "full_trajectory_flow_feasible": bool(flow_feas.success),
    "compact_matches_full_leaf_endpoints": bool(max_endpoint_difference < 1e-9),
    "compact_matches_full_random_support_functions": bool(max_objective_difference < 1e-9),
    "all_horizons_never_widen_leaf_intervals": all_not_wider,
    "at_least_one_strict_multi_horizon_gain": at_least_one_strict,
    "truth_contained_in_all_leaf_bounds": truth_contained,
    "global_robust_loss_covers_truth": bool(global_robust_upper + 1e-10 >= true_loss),
    "global_robust_loss_tighter_than_independent_interval_assembly": bool(
        global_robust_upper < independent_interval_upper - 1e-6
    ),
    "all_horizon_robust_loss_no_worse_than_last_horizon": bool(
        global_robust_upper <= last_robust_upper + 1e-10
    ),
    "largest_256_leaf_problem_under_five_seconds": bool(
        scaling_df.iloc[-1].solve_seconds < 5.0
    ),
}

verdict = (
    "PASS_TO_GENERAL_TREE_THEOREM_GATE"
    if all(checks.values())
    else "STOP_AND_REVISE_GLOBAL_FORMULATION"
)

summary = {
    "stage":"selective_refinement_global_compatibility_gate",
    "fixture_only":True,
    "seed":SEED,
    "gamma":GAMMA,
    "horizons":H,
    "tree_depth_edges":3,
    "leaves":len(leaves),
    "internal_nodes":len(children),
    "trajectory_groups":len(groups),
    "trajectory_flow_variables":nZ,
    "compact_leaf_variables":nL,
    "compact_inequalities":len(b_all),
    "max_true_sibling_odds_ratio":max_true_or,
    "max_crossing_mass_reconstruction_error":max_r_error,
    "max_full_vs_compact_leaf_endpoint_difference":max_endpoint_difference,
    "max_full_vs_compact_random_objective_difference":max_objective_difference,
    "mean_hierarchy_only_width":mean_hierarchy_width,
    "mean_last_horizon_width":mean_last_width,
    "mean_all_horizon_width":mean_all_width,
    "relative_multi_horizon_width_reduction":multi_horizon_reduction,
    "relative_all_horizon_vs_hierarchy_width_reduction":gamma_vs_hierarchy_reduction,
    "true_loss":true_loss,
    "last_horizon_robust_upper":last_robust_upper,
    "all_horizon_global_robust_upper":global_robust_upper,
    "independent_interval_upper":independent_interval_upper,
    "global_vs_independent_relative_tightening":1.0-global_robust_upper/independent_interval_upper,
    "largest_scaling_leaves":int(scaling_df.iloc[-1].leaves),
    "largest_scaling_inequalities":int(scaling_df.iloc[-1].inequalities),
    "largest_scaling_solve_seconds":float(scaling_df.iloc[-1].solve_seconds),
    "checks":checks,
    "verdict":verdict,
    "interpretation":(
        "Synthetic global validation only. A pass shows that, on this nontrivial depth-3 tree, "
        "one globally compatible trajectory-flow model exists and its identified leaf polytope "
        "is numerically identical to a compact subtree-mass LP. It also shows strict benefits "
        "from retaining global dependence and multiple horizons. The next gate is a proof for "
        "arbitrary finite rooted trees, not a claim of publication-level novelty."
    )
}

with open(OUTDIR/"global_compatibility_summary.json","w") as f:
    json.dump(summary,f,indent=2)

zip_name = "selective_refinement_global_compatibility_gate_results.zip"
with zipfile.ZipFile(zip_name,"w",zipfile.ZIP_DEFLATED) as zf:
    for p in OUTDIR.iterdir():
        zf.write(p, arcname=p.name)

print(json.dumps(summary, indent=2))
print("\nVERDICT:", verdict)
print("Results archive:", zip_name)

try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    pass


{
  "stage": "selective_refinement_global_compatibility_gate",
  "fixture_only": true,
  "seed": 20261004,
  "gamma": 4.0,
  "horizons": 3,
  "tree_depth_edges": 3,
  "leaves": 16,
  "internal_nodes": 12,
  "trajectory_groups": 121,
  "trajectory_flow_variables": 170,
  "compact_leaf_variables": 16,
  "compact_inequalities": 135,
  "max_true_sibling_odds_ratio": 3.0370370370370376,
  "max_crossing_mass_reconstruction_error": 5.551115123125783e-17,
  "max_full_vs_compact_leaf_endpoint_difference": 4.996003610813204e-16,
  "max_full_vs_compact_random_objective_difference": 5.551115123125783e-16,
  "mean_hierarchy_only_width": 0.31963059267737526,
  "mean_last_horizon_width": 0.11484074798142302,
  "mean_all_horizon_width": 0.10966039454569984,
  "relative_multi_horizon_width_reduction": 0.04510901859121619,
  "relative_all_horizon_vs_hierarchy_width_reduction": 0.6569152106901498,
  "true_loss": 2.9259390296809693,
  "last_horizon_robust_upper": 3.105688560511656,
  "all_horizon_global_r